# IDB (Inter-American Development Bank)

**Source:** https://data.iadb.org/dataset/project-procurement-bidding-notices-and-notification-of-contract-awards
Consultancy procurement notices from the IDB's open data platform.

**What this notebook does**

Queries the IDB's open procurement dataset for recent consultancy notices and uploads anything new to the unified Notion database.

**Filters applied**

- Notice type: General Procurement Notice and Expression of Interest
- Title keywords: Consultancy, Consultant, Consulting, Advisory, Consultoría, Consultora, Consultoria
- Language: English or Spanish only, detected from the notice title
- Time window: notices published in the last four days
- Deadline: already-expired notices are skipped
- Blocked keywords: excluded if the title or description matches any term in `Sources/blocked_words.py`

**How the fetch works**

1. Build a query against the dataset's SQL endpoint, filtering by notice type, title keyword and publication date. Everything matching comes back in one request, so there is no pagination.
2. Detect the language from the notice title and drop anything that is neither English nor Spanish
3. Work out the closing date, see the note below
4. Deduplicate against `idb_contract_titles.csv`, apply the blocklist, and upload the remainder to Notion

**Notes**

- **Deadlines from this source are unreliable.** The dataset regularly returns dates falling before the publication date, and placeholder values such as `2001-01-01`. The deadline is only trusted if it falls strictly after the publication date, otherwise the publication date plus two months is used. Closing dates on IDB entries should be treated as indicative.
- Notice type values in the source data have inconsistent trailing spaces, so matching is done on the start of the value rather than the whole thing.
- This is the only source that accepts Spanish as well as English.
- Fields the dataset does not publish, left blank or marked unavailable in Notion: contract value, CPV codes, client, employer website. The description is the project name, which is often blank.

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID = '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}


In [2]:
# Shared keyword blocklist (sources/ root) - added 2026-08-09 per Javiera's feedback
import sys
from pathlib import Path

BLOCKED_WORDS_PATH = Path("../blocked_words.py")
if not BLOCKED_WORDS_PATH.exists():
    raise FileNotFoundError(f"Could not find {BLOCKED_WORDS_PATH.resolve()}")

sys.path.insert(0, str(BLOCKED_WORDS_PATH.parent.resolve()))
from blocked_words import is_blocked, blocked_keyword_hits


In [3]:
import requests
import pandas as pd
import json
import html
import os
import time
import langid
from datetime import datetime, timezone, timedelta
from dateutil import parser as _dateparser

IDB_SQL_API_URL = "https://data.iadb.org/api/action/datastore_search_sql"
IDB_RESOURCE_ID = "856aabfd-2c6a-48fb-a8b8-19f3ff443618"

# English terms confirmed against real data (Consultancy, Consultant) + Consulting/Advisory
# (not yet verified) — same base list as the OppsLink notebook. Spanish terms added here only,
# since Notion accepts Spanish contracts too (per Alex, 2026-07-31).
IDB_TITLE_TERMS = [
    "Consultancy", "Consultant", "Consulting", "Advisory",
    "Consultoría", "Consultora", "Consultoria",
]

# Only these two languages pass the filter.
SUPPORTED_IDB_LANGUAGES = {"en", "es"}
LANG_DISPLAY_NAME = {"en": "English", "es": "Spanish"}


def clean_description(description):
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
    return ' '.join(cleaned.split()).strip()


def build_idb_query(days_back=4):
    """
    4-day rolling window on publicationdate, same convention as UNGM/OppsLink — covers
    the Friday-to-Monday gap on the Mon-Fri schedule. Deduping against existing_titles
    below covers any overlap between runs.
    """
    cutoff = (datetime.now(timezone.utc) - timedelta(days=days_back)).strftime("%Y-%m-%d")
    title_clause = " OR ".join([f"noticetitle ILIKE '%{t}%'" for t in IDB_TITLE_TERMS])
    sql = (
        'SELECT noticeid, type, countryname, noticetitle, projectname, loannumber, '
        'proyecturl, documenturl, publicationdate, deadline '
        f'FROM "{IDB_RESOURCE_ID}" '
        "WHERE (type LIKE 'GENERAL%' OR type LIKE 'EOI%') "
        f"AND ({title_clause}) "
        f"AND publicationdate >= '{cutoff}' "
        "ORDER BY publicationdate DESC"
    )
    return sql


def fetch_idb_notices(days_back=4):
    """IDB's SQL endpoint returns everything matching in one call — no pagination needed here."""
    sql = build_idb_query(days_back=days_back)
    for attempt in range(5):
        try:
            r = requests.get(IDB_SQL_API_URL, params={"sql": sql}, timeout=30)
            r.raise_for_status()
            data = r.json()
            if data.get("success"):
                return data["result"]["records"]
            print(f"⚠️ IDB API returned success=False: {data}")
            return []
        except Exception as e:
            print(f"⚠️ IDB fetch attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return []


def detect_idb_language(title: str):
    """Top langid guess, taken as-is — same approach as OppsLink notebook."""
    try:
        lang, _ = langid.classify(title)
        return lang
    except Exception:
        return None


def validate_idb_deadline(publicationdate_str, deadline_str, fallback_months=2):
    """
    IDB's `deadline` field is unreliable — sometimes before the publish date, sometimes a
    legacy placeholder (2001-01-01 / 2011-01-01), sometimes blank. Only trust it if it's
    strictly after the publish date; otherwise fall back to publish-date + N months, same
    convention as EU Commission / the OppsLink notebook. Returns (deadline_date, used_fallback).
    """
    pub_dt = None
    if publicationdate_str:
        try:
            pub_dt = _dateparser.parse(publicationdate_str)
        except Exception:
            pub_dt = None
    if pub_dt is None:
        pub_dt = datetime.now(timezone.utc)

    deadline_dt = None
    if deadline_str:
        try:
            deadline_dt = _dateparser.parse(deadline_str)
        except Exception:
            deadline_dt = None

    if deadline_dt and deadline_dt.date() > pub_dt.date():
        return deadline_dt.date(), False

    fallback_date = (pub_dt + timedelta(days=30 * fallback_months)).date()
    return fallback_date, True


def is_idb_expired(deadline_date) -> bool:
    if not deadline_date:
        return False
    return deadline_date < datetime.now(timezone.utc).date()


def build_idb_description(notice):
    """Just the project name — falls back to "" (which becomes "Not Disclosed" via the
    upload cell's default) when projectname is blank, which is common in this dataset."""
    return (notice.get("projectname") or "").strip()


In [4]:
# 1) Load already-uploaded titles to avoid duplicates
csv_path = "idb_contract_titles.csv"
existing_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Title" in prev.columns:
            existing_titles = set(prev["Title"].dropna().astype(str).str.strip().str.lower())
    except Exception as e:
        print("Warning: could not read", csv_path, ":", e)

# 2) Fetch + filter (English or Spanish, not expired) + parse
idb_notices = fetch_idb_notices(days_back=4)
print(f"Fetched {len(idb_notices)} candidate IDB notices")

extracted_data = []
skipped_unsupported_language = 0
skipped_expired = 0

for notice in idb_notices:
    title = (notice.get("noticetitle") or "").strip()
    if not title:
        continue

    if title.strip().lower() in existing_titles:
        continue

    lang = detect_idb_language(title)
    if lang not in SUPPORTED_IDB_LANGUAGES:
        skipped_unsupported_language += 1
        print(f"🌐 Skipping unsupported language ({lang}): {title}")
        continue

    deadline_date, used_fallback = validate_idb_deadline(notice.get("publicationdate"), notice.get("deadline"))

    if is_idb_expired(deadline_date):
        skipped_expired += 1
        print(f"🚫 Skipping expired: {title} (deadline {deadline_date})")
        continue

    description_for_block_check = build_idb_description(notice)
    if is_blocked(title, description_for_block_check):
        hits = blocked_keyword_hits(title, description_for_block_check)
        print(f"⛔ Skipping blocked keyword ({', '.join(hits)}): {title}")
        continue

    extracted_data.append({
        "closing_date": deadline_date.isoformat() if deadline_date else None,
        "country": notice.get("countryname") or "Regional",
        "client": "",           # no distinct buyer/agency field in this dataset
        "client_link": "",
        "link": notice.get("documenturl") or notice.get("proyecturl") or "",
        "title": title,
        "description": build_idb_description(notice),
        "value": "Unavailable",
        "cpv_codes": "Not Applicable",
        "language": LANG_DISPLAY_NAME.get(lang, lang),
    })

print(f"✅ {len(extracted_data)} new contracts ready for Notion upload "
      f"(skipped {skipped_unsupported_language} non-English/Spanish, {skipped_expired} expired)")


Fetched 1 candidate IDB notices


✅ 1 new contracts ready for Notion upload (skipped 0 non-English/Spanish, 0 expired)


### Upload to Notion

In [5]:
def create_page(properties: dict) -> bool:
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers_notion, json=payload, timeout=60)
    if not res.ok:
        print("\u274c Notion error:", res.status_code, res.text[:500])
        return False
    print(f"\u2705 Page created: {properties['Name']['title'][0]['text']['content']}")
    return True


def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default


def _safe_iso(dt_str):
    if not dt_str:
        return None
    try:
        parsed = _dateparser.parse(dt_str)
        return parsed.isoformat()
    except Exception:
        return None


now_iso = datetime.now(timezone.utc).isoformat()
new_titles_for_csv = []

# Upload newest first, matching the other Notion notebooks' convention
for contract in list(reversed(extracted_data)):
    name = _safe_str(contract.get("title"))[:1000]
    if not name:
        continue

    closing_date_iso = _safe_iso(contract.get("closing_date"))

    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": _safe_str(contract.get("cpv_codes"))[:2000]}}]},
        "Client": {"rich_text": [{"text": {"content": _safe_str(contract.get("client"), "Not Disclosed")[:2000]}}]},
        "Contract Link": {"url": _safe_str(contract.get("link")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": _safe_str(contract.get("description"), "Not Disclosed")[:2000]}}]},
        "Employer Website": {"url": _safe_str(contract.get("client_link")) or None},
        "Language": {"rich_text": [{"text": {"content": _safe_str(contract.get("language"))[:2000]}}]},
        "Location": {"rich_text": [{"text": {"content": _safe_str(contract.get("country"))[:2000]}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": _safe_str(contract.get("value"), "Unavailable")[:2000]}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "IDB"}},
    }

    try:
        success = create_page(props)
        if success:
            new_titles_for_csv.append({"Title": name})
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# Save newly uploaded titles to CSV for next run's dedup
if new_titles_for_csv:
    new_df = pd.DataFrame(new_titles_for_csv, columns=["Title"])
    header_needed = not os.path.exists(csv_path)
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"\u2705 Uploaded {len(new_titles_for_csv)} new IDB contracts to Notion.")


✅ Page created: CONSULTANT FOR THE DEVELOPMENT OF SYSTEM DESIGN DOCUMENT FOR THE NATIONAL IDENTIFICATION AND REGISTRATION AUTHORITY (NIRA)
✅ Uploaded 1 new IDB contracts to Notion.
